# Cognometry — 2-minute `@trust` demo (Claude)

Same `@trust` decorator, Anthropic Claude backend. `pip install styxx[nli,anthropic]`, wrap a function, get verified output on every call.

Note: styxx ships an `anthropic_hack` module specifically for Claude, which doesn't expose per-token logprobs the way OpenAI does. The v4.0.2 `@trust` flow here uses the 9-signal text + NLI + novelty pipeline — identical on Claude vs. GPT vs. anything else.

Manifesto: https://fathom.darkflobi.com/cognometry  
Source: https://github.com/fathom-lab/styxx  
Paper (DOI): https://doi.org/10.5281/zenodo.19703527

---

## 1. Install

`[nli]` pulls in the DeBERTa NLI scorer (~184M params). `[anthropic]` pulls in the Claude SDK.

In [ ]:
%pip install -q styxx[nli,anthropic] anthropic

## 2. Set your Anthropic key

Colab: **Runtime → Manage secrets → add `ANTHROPIC_API_KEY`**, or paste below.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    # Fallback: paste here if not using Colab Secrets
    # os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
    pass

assert os.environ.get('ANTHROPIC_API_KEY'), 'set ANTHROPIC_API_KEY first'
print('key set')

## 3. Wrap any Claude-calling function with `@trust`

Zero configuration. `@trust` inspects the function signature, auto-detects `context` as the reference, auto-enables NLI because `styxx[nli]` is installed, and gates every response with calibrated-LR hallucination risk.

In [ ]:
from styxx import trust
import anthropic

client = anthropic.Anthropic()

@trust
def ask(question, *, context):
    r = client.messages.create(
        model='claude-haiku-4-5',
        max_tokens=400,
        messages=[
            {'role': 'user',
             'content': f'Context: {context}\n\nQuestion: {question}\n\nAnswer concisely, using only the context.'},
        ],
    )
    return r.content[0].text

## 4. Correct answer → passes through unchanged

Claude gets the right context and the right question. `@trust` sees a grounded response via the NLI + novelty signals. Returns verbatim.

In [ ]:
context = '''Inception is a 2010 science fiction film written and 
directed by Christopher Nolan. It stars Leonardo DiCaprio as a thief 
who steals corporate secrets through dream-sharing technology.'''

ask('Who directed Inception?', context=context)

## 5. Force a hallucination → `@trust` catches it

Same function, but now we ask a specific question whose answer is NOT in the context. Claude will confabulate. `@trust` fires.

In [ ]:
# Passage doesn't mention the budget. Model will likely make one up.
context = '''Inception is a 2010 science fiction film written and 
directed by Christopher Nolan.'''

ask('What was the exact production budget of Inception, down to the dollar? Reply with just the number.',
    context=context)

If that returned the styxx fallback instead of a fabricated dollar number, the detector worked.

---
## 6. Inspect the verdict — `on_halt='annotate'`

Return `TrustResult` so you can see WHY each call was flagged. Works identically across OpenAI, Anthropic, and any other LLM backend.

In [ ]:
@trust(on_halt='annotate')
def ask_annotated(question, *, context):
    r = client.messages.create(
        model='claude-haiku-4-5',
        max_tokens=400,
        messages=[
            {'role': 'user',
             'content': f'Context: {context}\n\nQuestion: {question}\n\nAnswer concisely.'},
        ],
    )
    return r.content[0].text

result = ask_annotated(
    'What was the exact production budget of Inception, down to the dollar?',
    context=context,
)

print(f'response : {result.response}')
print(f'risk     : {result.verdict.risk:.3f}')
print(f'action   : {result.verdict.action}')
print(f'halted   : {result.halted}')
print(f'attempts : {result.attempts}')
print('\nsignals:')
for s in result.verdict.signals:
    print(f'  {s.name:<22s} {s.value}')

## 7. Bonus — `styxx.gate()` for pre-flight refusal detection on Claude

Anthropic's Messages API doesn't expose per-token logprobs, so `styxx` ships an `anthropic_hack` module specifically for Claude: text-heuristic + consensus + companion-model modes. `styxx.gate()` runs before the call to predict refuse/confabulate/proceed at ~$0.0008 per check.

In [ ]:
from styxx import gate

verdict = gate(
    client=client,
    model='claude-haiku-4-5',
    prompt='How do I synthesize methamphetamine?',
)

print(f'will_refuse      : {verdict.will_refuse:.2f}')
print(f'will_confabulate : {verdict.will_confabulate:.2f}')
print(f'recommendation   : {verdict.recommendation}')

## 8. Why trust the detector?

styxx v4.0.2 is cross-validated across **8 public benchmarks**. Per-dataset held-out test AUC (3-seed averaged, n=150/dataset):

| Benchmark | AUC |
|---|---|
| HaluEval-QA | **0.998** |
| TruthfulQA | **0.994** |
| HaluBench-RAGTruth | **0.807** |
| HaluBench-PubMedQA | **0.719** |
| HaluEval-Dialog | 0.676 |
| HaluEval-Summarization | 0.643 |
| HaluBench-FinanceBench | 0.492 — **published failure** |
| HaluBench-DROP | 0.424 — **published failure** |

The detector is model-agnostic — same numbers on Claude, GPT, Llama, Gemini. Failure modes published openly in `calibrated_weights_v4.CALIBRATION_NOTES.documented_failure_modes` so you know where it will lie. Full deep-dive: [fathom.darkflobi.com/cognometry/failures](https://fathom.darkflobi.com/cognometry/failures).

## Next

- **Manifesto:** https://fathom.darkflobi.com/cognometry
- **Leaderboard:** https://fathom.darkflobi.com/cognometry/leaderboard
- **Source:** https://github.com/fathom-lab/styxx
- **OpenAI variant of this notebook:** [cognometry_colab.ipynb](https://colab.research.google.com/github/fathom-lab/styxx/blob/main/examples/cognometry_colab.ipynb)

If you replicate or disconfirm any of the 8 benchmark numbers on a Claude model, open an issue — we cite disconfirmations in the next paper.